In [0]:
%sql
select * from data_sentinals.bronze.raw_customers;

In [0]:
CONFIG_PATH = "/Workspace/Users/{current_user}/data-sentinels/Olist_DE_Practise_Jobs/utilities/ingestion_config.json"
import json
try:
    with open(CONFIG_PATH, "r") as f:
        pipeline_config = json.load(f)
except Exception as e:
    print(f"Error loading config file: {e}")

In [0]:
# customer segmentation
"""
Purpose : customer segmentation
Needed files : Customer id, Order id, Order value (at item level, can aggregate at order level), Order status, Order date, Customer state, Product categories (at high level)
Approach : to segment the customer we have 3 categories
1. Customer Lifecycle value : For this we collect the last 12 month data, the anchor date is fixed based on the max date of the order date as the data is not upto date. (categories are high value, medium value, low value)
2. Volume of orders : Number of products/orders items purchased by the customer in past 12 months. Based on the EDA we will decide on the threshold - determined as High Volume, Medium Volume, Low Volume
3. Payment type : Identify the primary method of payment for the customer and based on the purchase value for each customer ie if a customer does tranasactions $100 , $50 via credit card and cash respectively then the primary mode will be credit card. If they are same we can latest payment method. This will go as a filter into the dashboard where we can call this as primary payment method and identify percentage of customers in each of the categories.

Final Classification:
If high vol and high value : Then High Value Customer
IF low vol and high value : Then high value customer
If high vol and medium value : Then Medium Value Customer
If medium vol and medium value : then medium Value Customer
If low vol and low value : Then low value customer
If low vol and medium value : Then Meidum value customer
If high vol and low value : Then Low Value Customer

"""

# Customer Segmentation Strategy Document

**Purpose:** To segment historical e-commerce customers into actionable personas based on their purchasing behavior, enabling targeted marketing campaigns and dynamic dashboard filtering.

**Required Data:**
* **Customer details:** Customer ID, Customer State
* **Order details:** Order ID, Order Date, Order Status
* **Financials:** Order Value (Item price + freight)
* **Payments:** Payment Type, Payment Value, Payment Sequential/Date
* *(Note: Product catalogs and categories are excluded to prioritize frequency and monetary metrics.)*

**Data Preparation & Timeline Logic:**
1.  **Exclusions:** Filter out all orders with an `order_status` of "canceled" or "unavailable".
2.  **The L12M Window:** Because the dataset is historical, establish an "Anchor Date" by finding the absolute maximum `order_date` in the entire dataset. The active analysis window is exactly 12 months prior to this Anchor Date. Customers with no purchases in this window are tagged as "Inactive/Historical".

**Segmentation Dimensions (The 3 Core Rules):**

1.  **Monetary Value (L12M Spend):** Sum the total valid purchase value per customer over the last 12 months. Split into tiers based on percentile ranking (e.g., Top 15%, Middle 55%, Bottom 30%):
    * High Value
    * Medium Value
    * Low Value

2.  **Order Volume (Loyalty/Frequency):** Count the total number of unique orders placed by the customer in the L12M window. Thresholds will be determined strictly via EDA (Exploratory Data Analysis), but will generally fall into:
    * High Volume (Frequent repeat buyers)
    * Medium Volume (Occasional repeat buyers)
    * Low Volume (One-time buyers)

3.  **Primary Payment Preference:** Identify the preferred payment method by summing the total *purchase value* per payment type for each customer. 
    * The payment type with the highest total value becomes the primary mode.
    * *Tie-breaker:* If values are identical, default to the most recently used payment method.

**Final Classification (The Persona Matrix):**
By intersecting the Monetary Value and Order Volume dimensions, we generate actionable, distinct customer personas:

* **High Value + High Volume:** *VIP Loyalists* (High-ticket, frequent shoppers. Target: VIP Rewards/Loyalty programs)
* **High Value + Low/Med Volume:** *Premium One-Offs* (Bought 1-2 very expensive items. Target: Cross-sell accessories/warranties)
* **Medium Value + High Volume:** *Frequent Core Shoppers* (Reliable, steady buyers. Target: Subscription models or bundle deals)
* **Medium Value + Low Volume:** *Casual Core Shoppers* (Average spend, rare visits. Target: "We miss you" discount campaigns)
* **Low Value + High Volume:** *Bargain Hunters* (Frequent checkouts, very low basket size. Target: Minimum-spend free shipping offers)
* **Low Value + Low Volume:** *Occasional Low-Value* (Bought 1 cheap item. Target: General brand awareness)

**Dashboard Integration:**
* **Primary Metrics:** The Final Classification Personas.
* **Global Filters:** Customer State, Order Year, and Primary Payment Preference will not be embedded in the persona logic. Instead, they will act as dynamic global filters in the BI dashboard to allow stakeholders to slice the personas by region, time, and payment habit.

In [0]:
%sql
-- EDA for identifying the threshold for param 2 order volume

select * from data_sentinals.bronze.raw_orders
where customer_id ='8d50f5eadf50201ccdcedfb9e2ac8455'; -- order details
-- select * from data_sentinals.bronze.raw_customers limit 10; -- customer details


In [0]:
%sql
select customer_unique_id, count(customer_id) from data_sentinals.bronze.raw_customers 
group by 1
order by 2 desc;

In [0]:
%sql
select * from data_sentinals.bronze.raw_customers
where customer_unique_id = '8d50f5eadf50201ccdcedfb9e2ac8455'; -- 1bd3585471932167ab72a84955ebefea

select * from data_sentinals.bronze.raw_orders
where customer_id = '1bd3585471932167ab72a84955ebefea'; -- b850a16d8faf65a74c51287ef34379ce

select * from data_sentinals.bronze.raw_order_items
where order_id = 'b850a16d8faf65a74c51287ef34379ce';


select * from data_sentinals.bronze.raw_order_payments
where order_id = 'b850a16d8faf65a74c51287ef34379ce';

In [0]:
%sql
select max(order_purchase_timestamp), min(order_purchase_timestamp) from data_sentinals.silver.last_12months_orders limit 10;
  

In [0]:
%sql
select * from data_sentinals.silver.stg_cust_percentile ;

In [0]:
%sql
with agg_cust as (
  select customer_unique_id, sum(total_price) as unique_val
  from data_sentinals.silver.stg_last_12months_orders 
  group by 1
)
select count(*) from agg_cust;

In [0]:
%sql
select *, a.spend_percentile*100 from data_sentinals.silver.stg_last_12months_orders a 
order by spend_percentile desc
limit 10;
-- group by 1
-- having count(*) >1;
 /*
Get the customer unique id - this will be right grouping value
total order value at unique id level
*/

In [0]:
%sql
select customer_unique_id, count(order_id) as order_vol
from data_sentinals.silver.stg_last_12months_orders
group by 1
order by 2 desc;

In [0]:
%sql
select distinct customer_persona from data_sentinals.gold.customer_segmentation;
-- where volume_segment = 'Medium';

In [0]:
%sql
select count(*) from data_sentinals.silver.stg_cust_vol_agg;
select * from data_sentinals.silver.stg_cust_percentile;

In [0]:
%sql
select * from data_sentinals.silver.stg_cust_percentile
where spend_percentile between 0.6 and 0.7
order by spend_percentile desc;

In [0]:
%sql
select * from data_sentinals.gold.customer_segmentation;